# IBM HR Analytics — Employee Attrition EDA

**Business Question:** Why are employees leaving, and which profiles are most at risk?

**Dataset:** IBM HR Analytics (1,470 employees · 35 features)  
**Source:** [Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

---
**Sections:**
1. Setup & Data Loading
2. Dataset Overview
3. Target Variable Analysis
4. Numerical Features
5. Categorical Features
6. Correlation Analysis
7. Business Conclusions

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Color palette: left = stayed, right = left company
COLORS = {'No': '#4C72B0', 'Yes': '#DD8452'}

In [ ]:
DATA_PATH = 'data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
# Basic info
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

In [ ]:
# Drop columns with zero variance (same value for all employees)
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f'Constant columns removed: {constant_cols}')
df.drop(columns=constant_cols, inplace=True)

# Summary statistics
df.describe()

## 3. Target Variable — Attrition Rate

In [ ]:
attrition_counts = df['Attrition'].value_counts()
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100

print('=== Attrition Summary ===')
for label in ['No', 'Yes']:
    print(f'  {label}: {attrition_counts[label]:,} employees ({attrition_pct[label]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(attrition_counts.index, attrition_counts.values,
            color=[COLORS[k] for k in attrition_counts.index])
axes[0].set_title('Employee Count by Attrition')
axes[0].set_xlabel('Attrition')
axes[0].set_ylabel('Number of Employees')
for i, (label, val) in enumerate(attrition_counts.items()):
    axes[0].text(i, val + 10, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(attrition_counts.values, labels=attrition_counts.index,
            colors=[COLORS[k] for k in attrition_counts.index],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Attrition Rate')

plt.suptitle('Overall Employee Attrition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Numerical Features

In [ ]:
# Key numerical variables to explore
num_cols = ['Age', 'MonthlyIncome', 'DistanceFromHome',
            'TotalWorkingYears', 'YearsAtCompany', 'YearsSinceLastPromotion']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in COLORS.items():
        subset = df[df['Attrition'] == label][col]
        axes[i].hist(subset, alpha=0.6, bins=25, color=color, label=f'Attrition={label}')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Numerical Features by Attrition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for clearer comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df.boxplot(column=col, by='Attrition', ax=axes[i],
               boxprops=dict(color='navy'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col)
    axes[i].set_xlabel('Attrition')

plt.suptitle('Distribution of Numerical Features by Attrition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mean comparison table: stayed vs left
mean_comparison = df.groupby('Attrition')[num_cols].mean().T
mean_comparison['Difference (%)'] = ((mean_comparison['Yes'] - mean_comparison['No']) / mean_comparison['No'] * 100).round(1)
mean_comparison.round(1)

## 5. Categorical Features

In [ ]:
def plot_attrition_rate(col, ax, title=None):
    """Plot attrition rate (%) for each category in a column."""
    rates = (df.groupby(col)['Attrition']
               .apply(lambda x: (x == 'Yes').mean() * 100)
               .sort_values(ascending=False))
    
    bars = ax.bar(rates.index.astype(str), rates.values,
                  color=sns.color_palette('muted', len(rates)))
    ax.set_title(title or col)
    ax.set_ylabel('Attrition Rate (%)')
    ax.set_xlabel('')
    ax.set_ylim(0, rates.max() * 1.3)
    
    # Add percentage labels
    for bar, val in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=9)
    
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

cat_cols = ['Department', 'JobRole', 'BusinessTravel', 'OverTime', 'MaritalStatus']
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    plot_attrition_rate(col, axes[i])

axes[-1].axis('off')  # Hide unused subplot
plt.suptitle('Attrition Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction scores vs attrition
satisfaction_cols = ['JobSatisfaction', 'EnvironmentSatisfaction',
                     'RelationshipSatisfaction', 'WorkLifeBalance']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(satisfaction_cols):
    plot_attrition_rate(col, axes[i], title=col.replace('Satisfaction', ' Satisfaction'))

plt.suptitle('Attrition Rate by Satisfaction Scores (1=Low, 4=High)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Encode Attrition as binary for correlation
df['Attrition_binary'] = (df['Attrition'] == 'Yes').astype(int)

# Select numeric columns and compute correlation with attrition
numeric_df = df.select_dtypes(include='number')
corr_with_attrition = (numeric_df.corr()['Attrition_binary']
                        .drop('Attrition_binary')
                        .sort_values()
                        .reset_index())
corr_with_attrition.columns = ['Feature', 'Correlation']

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#DD8452' if c > 0 else '#4C72B0' for c in corr_with_attrition['Correlation']]
ax.barh(corr_with_attrition['Feature'], corr_with_attrition['Correlation'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Attrition\n(orange = positive, blue = negative)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of most correlated numeric features
top_features = corr_with_attrition.nlargest(8, 'Correlation')['Feature'].tolist() + \
               corr_with_attrition.nsmallest(8, 'Correlation')['Feature'].tolist()
top_features.append('Attrition_binary')

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(numeric_df[top_features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Top Features Related to Attrition',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Business Conclusions

### Key Findings

*(Complete this section after running the analysis — replace the placeholders with your actual numbers)*

**1. Overall attrition rate is ~16%**  
Out of 1,470 employees, approximately 237 left the company — a significant cost given replacement costs of 50–200% of annual salary.

**2. Overtime is the strongest predictor**  
Employees who work overtime have an attrition rate ~3x higher than those who don't. This is the single most actionable lever for HR.

**3. Low monthly income drives turnover**  
Employees who left earned significantly less on average. Salary benchmarking and raises for bottom-quartile earners could reduce attrition.

**4. Young employees and early-career stages are highest risk**  
Employees aged 25–35 with fewer years at the company and low job levels show the highest turnover rates.

**5. Sales Representatives and Lab Technicians have critical attrition**  
These roles show the highest role-specific attrition — targeted retention programs are recommended.

---

### Recommended Actions for the Business

| Priority | Action | Expected Impact |
|----------|--------|-----------------|
| High | Limit mandatory overtime or compensate it better | Reduce attrition in overtime workers |
| High | Salary review for bottom 25% earners | Reduce income-driven turnover |
| Medium | Mentorship program for employees < 3 years at company | Improve early retention |
| Medium | Career path clarity for Sales Reps and Lab Technicians | Address role-specific turnover |
| Low | Work-life balance initiatives | Improve satisfaction scores |

---

### Next Steps
- Build a **classification model** (Logistic Regression / Random Forest) to predict individual attrition risk → see project `02_machine_learning/`
- Create an **interactive dashboard** to let HR managers explore these insights dynamically → see project `05_dashboard/`